# 01 · Ingesta y auditoría del ENASEMsimple

**Objetivo:** leer el Stata conservando códigos especiales, verificar identificadores y cobertura por ola.

**Preparación:** poner `simpleMHAS.dta` en `data/raw/` y ejecutar `pip install -e ".[notebooks,dev]"`.
**Regla:** nunca subir datos a GitHub ni guardarlos como salida del notebook.


In [ ]:
from tesis_enasem.config import RAW_DIR, ROUND_TO_YEAR
from tesis_enasem.io import load_stata
from tesis_enasem.panel import make_person_id, add_year
from tesis_enasem.eda import availability_by_wave
from tesis_enasem.missing import missing_audit
from tesis_enasem.validation import panel_quality
from tesis_enasem.variables import GRUPOS
import pandas as pd

raw = load_stata(RAW_DIR / 'simpleMHAS.dta', preserve_missing=True)
raw['id'] = make_person_id(raw)
raw = add_year(raw, ROUND_TO_YEAR)
print(f'Filas: {len(raw):,} · personas: {raw.id.nunique():,}')
print(panel_quality(raw))


## Tipos de entrevista por ronda
`tipent` 1/2: directa; 3/4: sustituta; 5: sobre fallecido. *No* eliminar estas filas antes del diagnóstico.


In [ ]:
display(pd.crosstab(raw['ano'], raw['tipent'], dropna=False))


## Valores perdidos con causa
`.f` = fallecimiento; `.p` = sustituto; `.w` = ítem no formulado en esa ola. Distinguir de no respuesta o saltos.


In [ ]:
variables = [v for grupo in GRUPOS.values() for v in grupo if v in raw.columns]
display(missing_audit(raw, variables).sort_values(['variable','tipo']).head(80))


## Cobertura por ronda
La disponibilidad no equivale a comparabilidad metodológica; contrastar con el diccionario antes de modelar.


In [ ]:
clave = ['imp_neto','edad','n_enf','n_sint_depr','n_abvd','recuerdo1','recuerdo2','orientacion','serial7','asiste_cursos']
display(availability_by_wave(raw, clave).pivot(index='variable',columns='ronda',values='n_disponible'))


**Registro pendiente:** anotar conteos verificados en `docs/RESULTADOS.md` y criterios de muestra en `docs/DECISIONES.md`; ver CP1.
